In [ ]:
import json
import sys # Import sys to handle potential encoding issues
import os  # Import os to handle file paths reliably
import re  # Import regular expressions for cleaning text
import string # Import string constants for punctuation

# --- Configuration ---
# Set the correct encoding for your system
FILE_ENCODING = 'utf-8'

# Define the directory where the data files are located
DATA_DIR = 'data'

# Define the base names of your JSONL files
ORIGINAL_FILE_BASENAME = 'original_data.jsonl' # Basename of the original file with answers
NEW_FILE_BASENAME = 'new_data.jsonl'           # Basename of the new file without answers
OUTPUT_FILE_BASENAME = 'output_data_matched_by_story_final.jsonl' # New output name

# Construct the full paths
ORIGINAL_JSONL_PATH = os.path.join(DATA_DIR, ORIGINAL_FILE_BASENAME)
NEW_JSONL_PATH = os.path.join(DATA_DIR, NEW_FILE_BASENAME)
OUTPUT_JSONL_PATH = os.path.join(DATA_DIR, OUTPUT_FILE_BASENAME)

# --- Key Definitions for Matching ---
# Key containing the story text in the ORIGINAL file (used to build the answer map)
ORIGINAL_STORY_KEY = 'STORY'

# Key containing the story text in the NEW file (used to look up in the answer map)
NEW_STORY_KEY = 'original_english'

# Key containing the answer in the ORIGINAL file
ANSWER_KEY = '答案\nANSWER'

# --- Markers for Cleaning New Story Text ---
STORY_MARKER_PREFIX = "###STORY\n"
END_OF_STORY_MARKER = "\n\n###QUESTION\n" # Marker indicating end of story in new file
# --- End Configuration ---

# --- Debugging Flags ---
PRINT_GENERAL_DETAILS = False # Print general details for first few lines? (Set to False to focus on compare)
MAX_GENERAL_PRINTS = 5        # Limit general prints if enabled

# --- Detailed Debugging Configuration ---
# Set to line 1 based on the examples provided
COMPARE_ORIGINAL_LINE_NUM = 1 # Line number in original_data.jsonl
COMPARE_NEW_LINE_NUM = 1      # Corresponding line number in new_data.jsonl
ENABLE_DETAILED_COMPARE = True  # Set to True to activate comparison
# --- End Detailed Debugging ---

def normalize_text(text):
    """
    Normalizes text for matching:
    - Converts to lowercase.
    - Removes punctuation.
    - Replaces all whitespace sequences with a single space.
    - Strips leading/trailing whitespace.
    """
    if not isinstance(text, str):
        return "" # Return empty string if input is not a string

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation (using string.punctuation)
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)

    # 3. Replace all whitespace (space, tab, newline, etc.) with a single space
    text = re.sub(r'\s+', ' ', text)

    # 4. Strip leading/trailing whitespace
    text = text.strip()

    return text

def clean_new_story(text, prefix_marker, end_marker):
    """
    Cleans the story text from the new file:
    - Removes the prefix marker.
    - Truncates the text before the end marker.
    """
    if not isinstance(text, str):
        return ""

    cleaned_text = text
    # Remove prefix if present
    if cleaned_text.startswith(prefix_marker):
        cleaned_text = cleaned_text[len(prefix_marker):]

    # Find the end marker and truncate
    end_index = cleaned_text.find(end_marker)
    if end_index != -1: # If the end marker is found
        cleaned_text = cleaned_text[:end_index] # Keep only the part before it

    return cleaned_text

# --- Global storage for detailed comparison ---
original_compare_data = {'raw': None, 'normalized': None}
# ---

def add_answers_to_jsonl(original_path, new_path, output_path,
                         original_story_key, new_story_key, answer_key,
                         prefix_marker, end_marker, encoding):
    """
    Reads original and new JSONL files, matches records based on *normalized* story text
    (after cleaning prefix and suffix from the new file's story),
    adds the answer from the original to the new records, and writes to an output file.
    Includes detailed comparison for specific lines.

    Args:
        original_path (str): Path to the original JSONL file (contains answers).
        new_path (str): Path to the new JSONL file (missing answers).
        output_path (str): Path to write the updated JSONL file.
        original_story_key (str): The key for the story text in the original file.
        new_story_key (str): The key for the story text in the new file.
        answer_key (str): The key field containing the answer in the original file.
        prefix_marker (str): The prefix string to remove from the start of the new story text.
        end_marker (str): The marker indicating the end of the story section in the new file.
        encoding (str): The file encoding to use (e.g., 'utf-8').
    """
    answer_map_by_normalized_story = {} # Map uses normalized keys now
    processed_count = 0
    skipped_count = 0
    general_prints_orig = 0
    general_prints_new = 0

    # Ensure the output directory exists
    output_dir = os.path.dirname(output_path)
    if output_dir and not os.path.exists(output_dir):
        try:
            os.makedirs(output_dir)
            print(f"Created output directory: {output_dir}")
        except OSError as e:
            print(f"Error creating output directory '{output_dir}': {e}")
            return

    # 1. Read the original file and store answers mapped by *normalized* story text
    print(f"Reading original data from: {original_path}")
    print(f"Mapping answers using Original Story Key: '{original_story_key}', Answer Key: '{answer_key}'")
    print("--- Normalizing original story text for mapping ---")
    try:
        with open(original_path, 'r', encoding=encoding) as infile:
            for i, line in enumerate(infile):
                current_line_num = i + 1
                try:
                    record = json.loads(line.strip())
                    story_value = record.get(original_story_key)
                    answer_value = record.get(answer_key)

                    # --- Detailed Compare - Store Original Data ---
                    if ENABLE_DETAILED_COMPARE and current_line_num == COMPARE_ORIGINAL_LINE_NUM:
                        print(f"\n>>> Storing data for detailed comparison from original line {COMPARE_ORIGINAL_LINE_NUM}")
                        original_compare_data['raw'] = story_value
                        original_compare_data['normalized'] = normalize_text(story_value) if story_value else ""
                    # --- End Detailed Compare Store ---

                    if story_value and answer_value:
                        normalized_story_key = normalize_text(story_value)
                        if not normalized_story_key: continue
                        if normalized_story_key in answer_map_by_normalized_story:
                             print(f"Warning: Duplicate normalized story found in original file (line {current_line_num}). Overwriting answer for normalized story starting with: '{normalized_story_key[:70]}...'")
                        answer_map_by_normalized_story[normalized_story_key] = answer_value

                except json.JSONDecodeError:
                    print(f"Warning: Skipping invalid JSON on line {current_line_num} in original file: {line.strip()}")
                except Exception as e:
                    print(f"Warning: An unexpected error occurred processing line {current_line_num} in original file: {e}")

        print(f"\nSuccessfully built answer map with {len(answer_map_by_normalized_story)} unique normalized stories from original file.")
        if not answer_map_by_normalized_story:
             print("WARNING: No answer records were loaded. Check original file format and keys.")
        # --- Print stored original compare data ---
        if ENABLE_DETAILED_COMPARE and original_compare_data['raw'] is not None:
             print("\n" + "="*20 + f" STORED ORIGINAL LINE {COMPARE_ORIGINAL_LINE_NUM} " + "="*20)
             print(f"  Raw Original Story:\n    '{original_compare_data['raw']}'")
             print(f"  Normalized Key Used:\n    '{original_compare_data['normalized']}'")
             print("="* (42 + len(str(COMPARE_ORIGINAL_LINE_NUM))))
        elif ENABLE_DETAILED_COMPARE:
             print(f"\nWARNING: Did not find or store data for original line {COMPARE_ORIGINAL_LINE_NUM}.")
        # ---

    except FileNotFoundError:
        print(f"Error: Original file not found at '{original_path}'.")
        return
    except Exception as e:
        print(f"An error occurred reading the original file: {e}")
        return

    # 2. Read the new file, match by *normalized* story (after cleaning prefix/suffix), add answer, write output
    print(f"\nProcessing new data from: {new_path}")
    print(f"Attempting to match using New Story Key: '{new_story_key}' (after cleaning prefix/suffix and normalizing)")
    print(f"Writing output to: {output_path}")
    try:
        with open(new_path, 'r', encoding=encoding) as infile, \
             open(output_path, 'w', encoding=encoding) as outfile:
            for i, line in enumerate(infile):
                current_line_num = i + 1
                try:
                    new_record = json.loads(line.strip())
                    story_value_raw = new_record.get(new_story_key)

                    # --- Detailed Compare - Process New Data ---
                    if ENABLE_DETAILED_COMPARE and current_line_num == COMPARE_NEW_LINE_NUM:
                        print("\n" + "="*20 + f" DETAILED COMPARE: NEW LINE {COMPARE_NEW_LINE_NUM} " + "="*20)
                        print(f"  Raw New Story ({new_story_key}):\n    '{story_value_raw}'")
                        # --- Apply Cleaning and Normalization for Comparison ---
                        cleaned_story_text = clean_new_story(story_value_raw, prefix_marker, end_marker)
                        normalized_lookup_story = normalize_text(cleaned_story_text)
                        # --- End Cleaning/Normalization ---
                        print(f"  Cleaned Story (Prefix/Suffix Removed):\n    '{cleaned_story_text}'")
                        print(f"  Normalized String for Lookup:\n    '{normalized_lookup_story}'")
                        # Compare with stored original normalized text
                        if original_compare_data['normalized'] is not None:
                             print(f"  Comparing with stored normalized original (line {COMPARE_ORIGINAL_LINE_NUM}):")
                             if normalized_lookup_story == original_compare_data['normalized']:
                                 print("    -> EXACT MATCH FOUND between normalized strings!")
                             else:
                                 print("    -> !!! NO EXACT MATCH between normalized strings.")
                        else:
                             print("  Cannot compare: Original comparison data was not stored.")
                        # Check map lookup result for this specific line
                        if normalized_lookup_story in answer_map_by_normalized_story:
                            print("  Lookup Result in Map: FOUND")
                        else:
                            print("  Lookup Result in Map: NOT FOUND")
                        print("="* (42 + len(str(COMPARE_NEW_LINE_NUM))) + "\n")
                    # --- End Detailed Compare Process ---

                    if story_value_raw:
                        # --- Apply Cleaning and Normalization for Matching ---
                        cleaned_story_value = clean_new_story(story_value_raw, prefix_marker, end_marker)
                        normalized_story_lookup = normalize_text(cleaned_story_value)
                        # --- End Cleaning/Normalization ---

                        if not normalized_story_lookup: # Don't try to match empty strings
                             skipped_count += 1
                             continue

                        if normalized_story_lookup in answer_map_by_normalized_story:
                            # Add the answer key/value from the map
                            new_record[answer_key] = answer_map_by_normalized_story[normalized_story_lookup]
                            outfile.write(json.dumps(new_record, ensure_ascii=False) + '\n')
                            processed_count += 1
                        else:
                            # Handle cases where a match wasn't found even after normalization
                            # print(f"Warning: No matching answer found for *normalized* story (line {current_line_num} in new file) starting with: '{normalized_story_lookup[:70]}...'. Skipping answer addition.")
                            skipped_count += 1
                    else:
                        # print(f"Warning: Skipping line {current_line_num} in new file. MISSING NEW STORY KEY: '{new_story_key}'.")
                        skipped_count += 1
                except json.JSONDecodeError:
                    print(f"Warning: Skipping invalid JSON on line {current_line_num} in new file: {line.strip()}")
                    skipped_count += 1
                except Exception as e:
                    print(f"Warning: An unexpected error occurred processing line {current_line_num} in new file: {e}")
                    skipped_count += 1

        print(f"\nProcessing Complete.")
        print(f" - Records successfully processed and written: {processed_count}")
        print(f" - Records skipped (missing key or no match): {skipped_count}")
        print(f"Output file created at: {output_path}")

    except FileNotFoundError:
        print(f"Error: New file not found at '{new_path}'.")
    except Exception as e:
        print(f"An error occurred processing the new file or writing the output file: {e}")

# --- Run the script ---
if __name__ == "__main__":
    # --- Pre-run Check for Colab ---
    # 1. Ensure 'original_data.jsonl' and 'new_data.jsonl' are in the 'data' folder.
    # 2. Comparison lines are set to 1 based on provided examples.
    # --------------------------------

    add_answers_to_jsonl(
        original_path=ORIGINAL_JSONL_PATH,
        new_path=NEW_JSONL_PATH,
        output_path=OUTPUT_JSONL_PATH,
        original_story_key=ORIGINAL_STORY_KEY,
        new_story_key=NEW_STORY_KEY,
        answer_key=ANSWER_KEY,
        prefix_marker=STORY_MARKER_PREFIX,
        end_marker=END_OF_STORY_MARKER, # Pass the end marker
        encoding=FILE_ENCODING
    )


Reading original data from: data/original_data.jsonl
Mapping answers using Original Story Key: 'STORY', Answer Key: '答案
ANSWER'
--- Normalizing original story text for mapping ---

>>> Storing data for detailed comparison from original line 1

Successfully built answer map with 1377 unique normalized stories from original file.

==================== STORED ORIGINAL LINE 1 ====================
  Raw Original Story:
    'Xiao Hong and Xiao Fang watch other children play on the playground. They chat about some interesting things happening on the playground and discuss going to the park together after school. Suddenly, Xiao Hong gives Xiao Fang a look and looks in the direction of the swing. Then, Xiao Hong smiles at Xiao Fang. Xiao Fang nods, and the two stand up. Xiao Mei sits on the swing and notices the interaction between Xiao Hong and Xiao Fang.'
  Normalized Key Used:
    'xiao hong and xiao fang watch other children play on the playground they chat about some interesting things hap